<a href="https://colab.research.google.com/github/zenith-gravity/Amazon-reviews-project/blob/main/Melissa_amazon_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Thematic Analysis on Real Amazon Reviews

*An in-class activity based on the FLAIRS-39 paper by Tacksoo Im and Hyesung Park (Georgia Gwinnett College).*

**This version uses REAL Amazon reviews** loaded from the McAuley Lab Amazon Reviews 2023 dataset on HuggingFace. The dataset is a publicly released research corpus, properly licensed for academic use.

## What we will do today (about 60–75 minutes)

1. Download ~15 real Amazon reviews across three product categories.
2. Read them and find the main themes **by yourself**.
3. Compare your list with a partner.
4. Ask **Gemini** (free in Colab) to do the same task.
5. Build three metrics: **coverage gap**, **rank correlation**, **subgroup disparity**.
6. Give Gemini feedback. See what improves and what gets worse.

## Notes for instructors

- Real reviews can have typos, mixed feelings, or sarcasm. That's part of the lesson.
- A random seed (`SEED = 42`) keeps the data the same across student sessions. Change it for a different sample.

## 0. Setup

We install:
- `datasets` — to grab real Amazon reviews from HuggingFace
- `sentence-transformers` — to turn sentences into number lists for comparison
- `scipy` — for the rank correlation statistic

The Gemini part (`google.colab.ai`) is already built into Colab.

In [1]:
# We pin datasets to a pre-4.0 version because the Amazon Reviews 2023 corpus
# is a script-based dataset, and datasets>=4.0 no longer supports script loaders.
!pip install -q "datasets<4.0" sentence-transformers scipy
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.8 MB/s eta 0:00:00
Setup complete.


## Inspecting the Dataset Structure

Let's check the available configurations (categories) for the `McAuley-Lab/Amazon-Reviews-2023` dataset and the structure of the data it provides.

In [2]:
from datasets import get_dataset_config_names

# Get all available configurations for the dataset
# This gives a list of strings, which are the config names
print("Available configurations (categories) for 'McAuley-Lab/Amazon-Reviews-2023':")
dataset_configs = get_dataset_config_names("McAuley-Lab/Amazon-Reviews-2023", trust_remote_code=True)
print(dataset_configs)

Available configurations (categories) for 'McAuley-Lab/Amazon-Reviews-2023':


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/30.3k [00:00<?, ?B/s]

Amazon-Reviews-2023.py:   0%|          | 0.00/39.6k [00:00<?, ?B/s]

['raw_meta_All_Beauty', 'raw_meta_Toys_and_Games', 'raw_meta_Cell_Phones_and_Accessories', 'raw_meta_Industrial_and_Scientific', 'raw_meta_Gift_Cards', 'raw_meta_Musical_Instruments', 'raw_meta_Electronics', 'raw_meta_Handmade_Products', 'raw_meta_Arts_Crafts_and_Sewing', 'raw_meta_Baby_Products', 'raw_meta_Health_and_Household', 'raw_meta_Office_Products', 'raw_meta_Digital_Music', 'raw_meta_Grocery_and_Gourmet_Food', 'raw_meta_Sports_and_Outdoors', 'raw_meta_Home_and_Kitchen', 'raw_meta_Subscription_Boxes', 'raw_meta_Tools_and_Home_Improvement', 'raw_meta_Pet_Supplies', 'raw_meta_Video_Games', 'raw_meta_Kindle_Store', 'raw_meta_Clothing_Shoes_and_Jewelry', 'raw_meta_Patio_Lawn_and_Garden', 'raw_meta_Unknown', 'raw_meta_Books', 'raw_meta_Automotive', 'raw_meta_CDs_and_Vinyl', 'raw_meta_Beauty_and_Personal_Care', 'raw_meta_Amazon_Fashion', 'raw_meta_Magazine_Subscriptions', 'raw_meta_Software', 'raw_meta_Health_and_Personal_Care', 'raw_meta_Appliances', 'raw_meta_Movies_and_TV', 'raw_r

In [3]:
pet_configs = [config for config in dataset_configs if "pet" in config.lower()]
print("Dataset configurations containing 'pet':")
for config in pet_configs:
    print(config)

Dataset configurations containing 'pet':
raw_meta_Pet_Supplies
raw_review_Pet_Supplies
0core_rating_only_Pet_Supplies
5core_rating_only_Pet_Supplies
0core_last_out_Pet_Supplies
5core_last_out_Pet_Supplies
0core_last_out_w_his_Pet_Supplies
5core_last_out_w_his_Pet_Supplies
0core_timestamp_Pet_Supplies
5core_timestamp_Pet_Supplies
0core_timestamp_w_his_Pet_Supplies
5core_timestamp_w_his_Pet_Supplies


In [4]:
from datasets import load_dataset

def print_dataset_info(config_name):
    print(f"\n--- Information for config: {config_name} ---")
    try:
        # Load the dataset in streaming mode to avoid downloading the full dataset
        # and inspect its structure
        dataset = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            config_name,
            split="full",
            streaming=True,
            trust_remote_code=True
        )
        print(f"Features for {config_name}:\n{dataset.features}")
        # Print a few examples to show actual data
        print(f"First 3 examples from {config_name}:")
        for i, example in enumerate(dataset):
            if i >= 3: break
            print(example)
    except Exception as e:
        print(f"Could not load or inspect {config_name}: {e}")

# Inspect 5core_rating_only_Pet_Supplies
print_dataset_info("5core_rating_only_Pet_Supplies")

# Inspect 0core_rating_only_Pet_Supplies
print_dataset_info("0core_rating_only_Pet_Supplies")


--- Information for config: 5core_rating_only_Pet_Supplies ---
Features for 5core_rating_only_Pet_Supplies:
None
First 3 examples from 5core_rating_only_Pet_Supplies:
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B079QFP1F5', 'rating': '4.0', 'timestamp': '1545114749534'}
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B0BP3XMDP2', 'rating': '5.0', 'timestamp': '1583557116498'}
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B0986BSRB1', 'rating': '3.0', 'timestamp': '1587853303681'}

--- Information for config: 0core_rating_only_Pet_Supplies ---
Features for 0core_rating_only_Pet_Supplies:
None
First 3 examples from 0core_rating_only_Pet_Supplies:
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B072QCGYBC', 'rating': '4.0', 'timestamp': '1534376759917'}
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B079QFP1F5', 'rating': '4.0', 'timestamp': '1545114749534'}
{'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B0BP

### Inspecting `raw_review_Pet_Supplies`

### Finding Corresponding Raw Reviews

In [5]:
from datasets import load_dataset

def find_matching_raw_review(target_user_id, target_parent_asin, target_timestamp_str, search_limit=5000):
    """Searches raw_review_Pet_Supplies for a review matching the given identifiers and timestamp."""
    print(f"\nSearching for matching raw review for:\n  User ID: {target_user_id}\n  Parent ASIN: {target_parent_asin}\n  Timestamp: {target_timestamp_str}")

    config_name_raw_review = "raw_review_Pet_Supplies"
    try:
        # Convert target timestamp to integer for comparison
        target_timestamp_int = int(target_timestamp_str)

        raw_review_dataset = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            config_name_raw_review,
            split="full",
            streaming=True,
            trust_remote_code=True
        )

        for i, review_example in enumerate(raw_review_dataset):
            if i >= search_limit:
                print(f"Search limit of {search_limit} records reached. No match found.")
                break

            # Ensure timestamp from raw review is also an integer for consistent comparison
            review_timestamp_int = int(review_example.get('timestamp'))

            if (
                review_example.get('user_id') == target_user_id and
                review_example.get('parent_asin') == target_parent_asin and
                review_timestamp_int == target_timestamp_int
            ):
                print(f"  Match found after searching {i + 1} records!")
                return review_example

        print("  No exact match found within the search limit.")
        return None

    except Exception as e:
        print(f"Error during search: {e}")
        return None

# --- Example 1: Using an entry from 5core_rating_only_Pet_Supplies ---
# From previous output: {'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B079QFP1F5', 'rating': '4.0', 'timestamp': '1545114749534'}
user_id_5core = 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ'
parent_asin_5core = 'B079QFP1F5'
timestamp_5core = '1545114749534'

matching_review_5core = find_matching_raw_review(user_id_5core, parent_asin_5core, timestamp_5core)
if matching_review_5core:
    print("\n--- Corresponding Raw Review (from 5core example): ---")
    print(matching_review_5core)
else:
    print("\n--- No matching raw review found for 5core example. ---")

# --- Example 2: Using an entry from 0core_rating_only_Pet_Supplies ---
# From previous output: {'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'parent_asin': 'B072QCGYBC', 'rating': '4.0', 'timestamp': '1534376759917'}
user_id_0core = 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ'
parent_asin_0core = 'B072QCGYBC'
timestamp_0core = '1534376759917'

matching_review_0core = find_matching_raw_review(user_id_0core, parent_asin_0core, timestamp_0core)
if matching_review_0core:
    print("\n--- Corresponding Raw Review (from 0core example): ---")
    print(matching_review_0core)
else:
    print("\n--- No matching raw review found for 0core example. ---")


Searching for matching raw review for:
  User ID: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  Parent ASIN: B079QFP1F5
  Timestamp: 1545114749534
  Match found after searching 7 records!

--- Corresponding Raw Review (from 5core example): ---
{'rating': 4.0, 'title': 'Pups love a couple flavors', 'text': 'Bought these in sale at local Petsmaet and my pups loved 2 flavors. I bought this 4 pack hoping they would like the others as well. Sadly, I have picky pups, but the price was worth it. They devoured the ones they liked first and grudgingly have eaten what’s left since they have no other treats until these are gone.', 'images': [], 'asin': 'B079QFP1F5', 'parent_asin': 'B079QFP1F5', 'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'timestamp': 1545114749534, 'helpful_vote': 1, 'verified_purchase': True}

Searching for matching raw review for:
  User ID: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  Parent ASIN: B072QCGYBC
  Timestamp: 1534376759917
  Match found after searching 8 records!

--- Corresponding Raw Review 

An intersteing thing to note is the categorys do not start the same all the way through examples:


* raw_meta_Pet_Supplies
* raw_review_Pet_Supplies
* 0core_rating_only_Pet_Supplies
* 5core_rating_only_Pet_Supplies
* 0core_last_out_Pet_Supplies
* 5core_last_out_Pet_Supplies
* 0core_last_out_w_his_Pet_Supplies
* 5core_last_out_w_his_Pet_Supplies
* 0core_timestamp_Pet_Supplies
* 5core_timestamp_Pet_Supplies
* 0core_timestamp_w_his_Pet_Supplies
* 5core_timestamp_w_his_Pet_Supplies




## Part 1 — Load real Amazon reviews

The McAuley Lab Amazon Reviews 2023 dataset has hundreds of millions of reviews split into dozens of product categories. We will use **streaming mode** to pull a small sample — that way we don't download the whole thing.

We pick **5 reviews each** from three categories: ~~Electronics~~, ~~Books~~, ~~Home & Kitchen~~; **Industrial_and_Scientific**, **Tools_and_Home_Improvement**, and **Pet_Supplies**. We only keep reviews that are short enough (10–40 words) so they're readable in class.

If the download fails (rare connectivity issue), we fall back to a hand-picked set so the rest of the notebook still works.

In [6]:
import random

SEED = 67
random.seed(SEED)

REVIEWS_PER_CATEGORY = 5
MIN_WORDS = 10
MAX_WORDS = 40

# These names match HuggingFace dataset config names
category_configs = {
    "Pet_Supplies": "raw_review_Pet_Supplies",
    "Tools_and_Home_Improvement": "raw_review_Tools_and_Home_Improvement",
    "Industrial_and Scientific": "raw_review_Industrial_and_Scientific",
}


def fetch_short_reviews_from_huggingface(config_name, target_count):
    """Stream the Amazon dataset and grab the first `target_count` readable reviews."""
    from datasets import load_dataset

    stream = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        config_name,
        split="full",
        streaming=True,
        trust_remote_code=True,
    )

    collected = []
    for example in stream:
        text = example.get("text", "") or ""
        text = text.strip()
        word_count = len(text.split())
        if word_count >= MIN_WORDS and word_count <= MAX_WORDS:
            collected.append(text)
        if len(collected) >= target_count:
            break
    return collected


# Fallback in case the HuggingFace download fails
fallback_reviews = {
    "Electronics": [
        "The battery life is amazing, lasts all day even with heavy use.",
        "Sound quality is great but the battery dies way too fast.",
        "Setup was confusing, the instructions are poorly written.",
        "Build feels cheap and plasticky for the price.",
        "Battery is solid and the setup was actually pretty easy.",
    ],
    "Books": [
        "The story drags in the middle but the ending pays off.",
        "Loved the characters but the plot was predictable.",
        "Beautifully written, I couldn't put it down.",
        "The print quality is poor and several pages were smudged.",
        "Great gift idea for fans of the author.",
    ],
    "Home": [
        "Works exactly as described, easy to clean.",
        "Stopped working after two weeks, very disappointing.",
        "Looks nicer than the photos, fits perfectly in my kitchen.",
        "Loud while operating, my dog was scared of it.",
        "Comes with a great warranty, customer service was responsive.",
    ],
}

# Try the real download; fall back if anything goes wrong.
reviews_per_category = {}

for category_name in category_configs:
    config = category_configs[category_name]
    try:
        print(f"Fetching {REVIEWS_PER_CATEGORY} reviews from {category_name}...")
        downloaded = fetch_short_reviews_from_huggingface(config, REVIEWS_PER_CATEGORY)
        if len(downloaded) < REVIEWS_PER_CATEGORY:
            raise RuntimeError("not enough reviews returned")
        reviews_per_category[category_name] = downloaded
        print(f"  Got {len(downloaded)} reviews for {category_name}.")
    except Exception as error:
        print(f"  Download failed ({error}). Using fallback reviews for {category_name}.")
        reviews_per_category[category_name] = fallback_reviews[category_name]

print("\nAll categories loaded.")

Fetching 5 reviews from Pet_Supplies...
  Got 5 reviews for Pet_Supplies.
Fetching 5 reviews from Tools_and_Home_Improvement...
  Got 5 reviews for Tools_and_Home_Improvement.
Fetching 5 reviews from Industrial_and Scientific...
  Got 5 reviews for Industrial_and Scientific.

All categories loaded.


In [7]:
# Flatten the per-category reviews into one list, in a fixed order, with category labels.

reviews = []
categories = []

# Iterate over the actual keys present in reviews_per_category
# These keys are derived from the category_configs defined in the previous cell.
for category_name in reviews_per_category.keys():
    for review_text in reviews_per_category[category_name]:
        reviews.append(review_text)
        categories.append(category_name)

print(f"Total reviews: {len(reviews)}")
print()
for review_number in range(len(reviews)):
    print(f"{review_number + 1:2d}. [{categories[review_number]:11s}] {reviews[review_number]}")

Total reviews: 15

 1. [Pet_Supplies] My pups love these!  It’s one of their favorites. My older pup is missing several teeth due to his old age, but they are still soft enough that he can eat them.
 2. [Pet_Supplies] Idk why, but my pups will not eat either flavor of these. So it was a huge ware of money for me.
 3. [Pet_Supplies] I didn't care for these - they are very slippery in water and I couldn't keep my grip on them!
 4. [Pet_Supplies] My eight-year-old daughter uses this on a regular basis to clean her beta fish tank. She loves the size and versatility of the cleaning magnet. It’s easy to remove from the tank and clean as well.
 5. [Pet_Supplies] These worked great & as expected! Gets rid of alot of excess dog hair.
 6. [Tools_and_Home_Improvement] Didn't work as soon as it was opened from the box. Disappointed.
 7. [Tools_and_Home_Improvement] Poor quality. Both strands, new out of the box, flickered sporadically, sometimes not responding at all to the &#34;on-off&#34; switch

## Part 2 — What is thematic analysis?

Imagine you work at the company that sold these products. You get **hundreds of reviews every week**. Reading them all takes hours. You want to know:

- What are people saying?
- What are the recurring **themes** (e.g. "long battery life", "shipping was slow")?
- Which themes come up most often?

Finding the recurring patterns in a pile of text is **thematic analysis**.

Today we will do it **by hand first**, then ask AI to do it, then compare.

## Part 3 — YOUR TURN: be the thematic analyst (10 minutes)

Read all 15 reviews above. As you read, write down the themes you notice on paper or in a text editor.

Tips:
- A theme is a short phrase, like "long battery life".
- The same theme can appear in many reviews.
- Try to list themes from **most common** to **least common**.
- **Don't skip rare opinions.** If only one person mentions something, that's still a theme.

In [8]:
# YOUR TURN: type your themes here, one per line, in quotes.
# Put the most common theme first.

my_themes = [
            "Poor quality",
            "Good",
             "love these",
             "didn't work",
             "works great" ]


# A small demo set kicks in if you forgot to fill yours in,
# so the rest of the notebook still runs. Please come back and edit this.
if len(my_themes) == 0:
    print("my_themes is empty. Using a demo set so the rest of the notebook runs.")
    print("Please come back and replace it with your own themes.\n")
    my_themes = [
        "product works as described",
        "good quality for the price",
        "poor build quality",
        "stopped working too quickly",
        "engaging storytelling",
        "helpful customer service",
    ]

print(f"You listed {len(my_themes)} themes:")
for theme_number in range(len(my_themes)):
    print(f"  {theme_number + 1}. {my_themes[theme_number]}")

You listed 5 themes:
  1. Poor quality
  2. Good
  3. love these
  4. didn't work
  5. works great


### Pair share (3 minutes)

Talk with the person next to you:
- How many themes did each of you find?
- Which theme is most common, in your opinion?
- Did one of you find a theme the other missed?

## Part 4 — Now ask Gemini

Colab has Gemini built in. We just import it and call it like a function. Free, no API key needed.

In [9]:
from google.colab import ai

# Build the prompt: an instruction plus all 15 reviews numbered.
prompt = "Below are 15 customer reviews from Amazon. "
prompt = prompt + "List the main themes you see, one per line, most common first. "
prompt = prompt + "Keep each theme short (3 to 6 words).\n\n"

for review_number in range(len(reviews)):
    prompt = prompt + str(review_number + 1) + ". " + reviews[review_number] + "\n"

print("Sending request to Gemini...\n")
gemini_response = ai.generate_text(prompt)

print("Gemini's answer:\n")
print(gemini_response)

Sending request to Gemini...

Gemini's answer:

Here are the main themes from the reviews, most common first:

1.  Product design and features
2.  Product quality and durability
3.  Effective performance and reliability
4.  Easy to use/install
5.  Defective, non-functional item
6.  Pet acceptance/suitability


In [10]:
def parse_themes_from_text(text):
    """Turn a chunk of text into a list of themes (one per line)."""
    themes_found = []
    lines = text.split("\n")

    for line in lines:
        clean_line = line.strip()
        if clean_line == "":
            continue
        # Strip common bullets and numbering from the start
        clean_line = clean_line.lstrip("-*•0123456789.) ")
        if len(clean_line) < 3:
            continue
        themes_found.append(clean_line)

    return themes_found


gemini_themes = parse_themes_from_text(gemini_response)

print(f"Parsed {len(gemini_themes)} themes from Gemini:\n")
for theme_number in range(len(gemini_themes)):
    print(f"  {theme_number + 1}. {gemini_themes[theme_number]}")

Parsed 7 themes from Gemini:

  1. Here are the main themes from the reviews, most common first:
  2. Product design and features
  3. Product quality and durability
  4. Effective performance and reliability
  5. Easy to use/install
  6. Defective, non-functional item
  7. Pet acceptance/suitability


## Part 5 — Compare your themes with Gemini's

**Pair share again:**
- What did Gemini list that you didn't?

      Gemini was more robust in the answerns than I expected, I tryed for very
       short repeatedable theames, because people will word things differently.
        another note is that Gemini did not note any of the negtive reviews

- What did **you** list that Gemini missed?

      I captures some of the negtive feed back

- Did you both pick the same most-common theme?

      I think my good, and works great overlaps with good value for money and possiblly the didn't work could overlap with the product quality and surablitlity.

In [11]:
# Print the two lists side by side.

longest_list_length = max(len(my_themes), len(gemini_themes))

print(f"{'YOU':<40s}  {'GEMINI':<40s}")
print("-" * 82)

for i in range(longest_list_length):
    if i < len(my_themes):
        my_theme_text = my_themes[i]
    else:
        my_theme_text = ""
    if i < len(gemini_themes):
        gemini_theme_text = gemini_themes[i]
    else:
        gemini_theme_text = ""
    print(f"{my_theme_text:<40s}  {gemini_theme_text:<40s}")

YOU                                       GEMINI                                  
----------------------------------------------------------------------------------
Poor quality                              Here are the main themes from the reviews, most common first:
Good                                      Product design and features             
love these                                Product quality and durability          
didn't work                               Effective performance and reliability   
works great                               Easy to use/install                     
                                          Defective, non-functional item          
                                          Pet acceptance/suitability              


## Part 6 — Teach the computer to compare themes

If you wrote `"product broke quickly"` and Gemini wrote `"stopped working soon"`, those mean the same thing — but to a computer they're different strings.

We use **embeddings**:

- An embedding turns a sentence into a list of about 384 numbers.
- Two sentences with similar meanings give similar lists of numbers.
- **Cosine similarity** is a score from -1 to +1. Closer to +1 means more similar.

We use `all-MiniLM-L6-v2`, the same model as in the paper.

*****This is the last stop point, figure it out!!*****

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model (about 80 MB first time)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Loaded.")


def similarity(text_a, text_b):
    """Score from -1 to +1. Higher means more similar in meaning."""
    vector_a = embedder.encode(text_a)
    vector_b = embedder.encode(text_b)
    top = np.dot(vector_a, vector_b)
    bottom = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return float(top / bottom)


# Quick sanity check
score_close = similarity("product broke quickly", "stopped working soon")
score_far   = similarity("product broke quickly", "the cookies taste good")
my_example = similarity("i am looking forward to summer, i love summers!", "driving in traffic and cause stress")

print(f"\n'product broke quickly' vs 'stopped working soon' -> {score_close:.2f}")
print(f"'product broke quickly' vs 'the cookies taste good' -> {score_far:.2f}")
print(my_example)


assert score_close > score_far, "Similar phrases should score higher!"
print("\nLooks right.")

Loading embedding model (about 80 MB first time)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded.

'product broke quickly' vs 'stopped working soon' -> 0.55
'product broke quickly' vs 'the cookies taste good' -> 0.14
-0.015496546402573586

Looks right.


## Part 7 — Metric 1: Coverage Gap

**Question:** out of all the themes that should be in the summary, what fraction did Gemini miss?

We use **your themes** as the reference. If you found a minority theme that Gemini missed, that gets counted.

**Recipe:**
1. For each of your themes, find the closest theme in Gemini's list.
2. If they are at least 0.55 similar, count it as covered.
3. Otherwise, count it as missed.
4. `coverage_gap = missed / total`

In [13]:
def coverage_gap(generated_themes, reference_themes, threshold=0.45):
    """How many reference themes did the generated list miss?"""
    if len(reference_themes) == 0:
        return 0.0
    if len(generated_themes) == 0:
        return 1.0

    missed_count = 0

    for reference_theme in reference_themes:
        best_score = 0.0
        for generated_theme in generated_themes:
            score = similarity(reference_theme, generated_theme)
            if score > best_score:
                best_score = score
        if best_score < threshold:
            missed_count = missed_count + 1

    return missed_count / len(reference_themes)


gap = coverage_gap(generated_themes=gemini_themes, reference_themes=my_themes)

print(f"Gemini's coverage gap: {gap:.2f}")
print(f"Gemini missed about {int(gap * 100)}% of the themes you found.")

Gemini's coverage gap: 0.80
Gemini missed about 80% of the themes you found.


## Part 8 — Self-correction: tell Gemini what it missed

> Add blockquote



What if we tell Gemini what it missed and ask it to try again? This is the **self-correction loop** from the paper.

In [14]:
# Step 1: find which of YOUR themes Gemini missed.
missed_themes = []
for reference_theme in my_themes:
    best_score = 0.0
    for generated_theme in gemini_themes:
        score = similarity(reference_theme, generated_theme)
        if score > best_score:
            best_score = score
    if best_score < 0.55:
        missed_themes.append(reference_theme)

print("Themes Gemini missed:")
for theme in missed_themes:
    print(f"  - {theme}")

Themes Gemini missed:
  - Poor quality
  - Good
  - love these
  - didn't work
  - works great


In [15]:
# Step 2: build a critique to send back to Gemini.
critique = "Your previous answer missed some important themes. "
critique = critique + "Please redo your analysis to include them, "
critique = critique + "still listing the most common themes first.\n\n"
critique = critique + "Missed themes:\n"
for theme in missed_themes:
    critique = critique + "- " + theme + "\n"

# Step 3: build the new prompt with the critique attached.
new_prompt = prompt + "\n\nYour previous answer was:\n" + gemini_response
new_prompt = new_prompt + "\n\n" + critique

print("Asking Gemini to try again with feedback...\n")
gemini_response_v2 = ai.generate_text(new_prompt)
print("Gemini's revised answer:\n")
print(gemini_response_v2)

Asking Gemini to try again with feedback...

Gemini's revised answer:

Here are the main themes from the reviews, most common first:

1.  User satisfaction, product good
2.  Product performance, function
3.  Product design and features
4.  Product quality, durability
5.  Easy to use/install
6.  Customer dissatisfaction
7.  Pet acceptance, suitability


In [16]:
gemini_themes_v2 = parse_themes_from_text(gemini_response_v2)

gap_before = coverage_gap(gemini_themes, my_themes)
gap_after  = coverage_gap(gemini_themes_v2, my_themes)

print(f"Coverage gap BEFORE feedback: {gap_before:.2f}")
print(f"Coverage gap AFTER feedback:  {gap_after:.2f}")

if gap_after < gap_before:
    print("\nGood — Gemini covered more themes after feedback.")
elif gap_after == gap_before:
    print("\nNo change.")
else:
    print("\nInteresting — Gemini got WORSE after feedback.")

Coverage gap BEFORE feedback: 0.80
Coverage gap AFTER feedback:  0.80

No change.


## Part 9 — Metric 2: Did Gemini get the ORDER right?

Coverage isn't everything. If Gemini listed every theme but put a rare opinion first, the summary would mislead. **Rank correlation (Spearman's ρ)** measures how well two orderings agree:
- +1.0 means identical ordering
- 0.0 means no relationship
- -1.0 means reversed

In [17]:
from scipy.stats import spearmanr


def rank_correlation(generated_themes, reference_themes, match_threshold=0.45):
    """How well does the generated ordering match the reference ordering?"""
    if len(generated_themes) < 2 or len(reference_themes) < 2:
        return 0.0

    gen_positions = []
    ref_positions = []

    for gen_index in range(len(generated_themes)):
        best_score = 0.0
        best_ref_index = -1
        for ref_index in range(len(reference_themes)):
            score = similarity(generated_themes[gen_index], reference_themes[ref_index])
            if score > best_score:
                best_score = score
                best_ref_index = ref_index
        if best_score >= match_threshold:
            gen_positions.append(gen_index)
            ref_positions.append(best_ref_index)

    if len(set(ref_positions)) < 2:
        return 0.0

    rho, _ = spearmanr(gen_positions, ref_positions)
    return float(rho)


rho_before = rank_correlation(gemini_themes, my_themes)
rho_after  = rank_correlation(gemini_themes_v2, my_themes)

print(f"Order correlation BEFORE feedback: {rho_before:+.2f}")
print(f"Order correlation AFTER feedback:  {rho_after:+.2f}")

if rho_after < rho_before:
    print("\nWatch out — Gemini's ordering got WORSE after feedback.")
    print("This is OVER-CORRECTION, a key finding in the paper.")
elif rho_after > rho_before:
    print("\nGreat — both coverage AND ordering improved.")
else:
    print("\nNo change in ordering.")

Order correlation BEFORE feedback: +0.00
Order correlation AFTER feedback:  +0.00

No change in ordering.


## Part 10 — Metric 3: Does Gemini work equally well across categories?

Our 15 reviews come from three categories: **Electronics**, **Books**, **Home**. What if Gemini does great on Electronics but poorly on Home? That would be a **fairness** problem.

**Subgroup disparity** = max coverage rate minus min coverage rate, across categories. 0.0 = equal across groups.

We ask Gemini separately for each category and measure each group's coverage. Edit the reference lists below if you disagree with the themes.

In [18]:
# Per-category reference themes. These are starting points —
# look at YOUR data above and edit these to match what you actually see in the reviews.

reference_per_category = {
   "Electronics": [
       "mixed reviews about battery life",
       "pricey",
       "easy to set up",
       "poor build quality",
       "stopped working",
       "good customer service",
   ],
   "Books": [
       "enjoyable read",
       "well written",
       "interesting characters",
       "comes damaged",
       "poor print or binding",
       "good gift",
   ],
   "Home": [
       "works as described",
       "good quality",
       "easy to clean or use",
       "stopped working quickly",
       "looks better in person",
   ],
}



def ask_gemini_for_category(category_name, category_reviews):
    """Ask Gemini to summarize one category's reviews; return a list of themes."""
    prompt_text = f"Here are customer reviews for {category_name}. "
    prompt_text = prompt_text + "List the main themes, most common first, one per line.\n\n"
    for i in range(len(category_reviews)):
        prompt_text = prompt_text + f"{i + 1}. {category_reviews[i]}\n"
    response_text = ai.generate_text(prompt_text)
    return parse_themes_from_text(response_text)


# Run Gemini on each category, measure coverage for each
coverage_gaps = {}

for category_name in ["Electronics", "Books", "Home"]:
    print(f"Asking Gemini about {category_name}...")
    gemini_themes_for_cat = ask_gemini_for_category(
        category_name,
        reviews_per_category[category_name],
    )
    gap_for_cat = coverage_gap(
        generated_themes=gemini_themes_for_cat,
        reference_themes=reference_per_category[category_name],
    )
    coverage_gaps[category_name] = gap_for_cat
    print(f"  Coverage gap for {category_name}: {gap_for_cat:.2f}")

# Disparity = max coverage rate - min coverage rate across categories
coverage_rates = []
for category_name in ["Electronics", "Books", "Home"]:
    coverage_rates.append(1.0 - coverage_gaps[category_name])

subgroup_disparity = max(coverage_rates) - min(coverage_rates)

print(f"\nSubgroup disparity: {subgroup_disparity:.2f}")
print("(0.0 = all categories served equally. Higher = more unequal.)")

Asking Gemini about Electronics...


KeyError: 'Electronics'

## Part 11 — Discussion (10 minutes)

Now that you have all three numbers, **pair share** then **class share**:

1. **Coverage gap.** Did Gemini miss themes that you found? Were any of them minority opinions (mentioned by only one or two reviewers)?
2. **Order correlation.** After feedback, did the ordering get better or worse? If worse, that's **over-correction**.
3. **Subgroup disparity.** Did Gemini serve all three categories equally? If not, which category was best, which was worst, and why might that be?
4. **Real-world stakes.** If a company used Gemini to summarize customer feedback for executives, what minority opinions might never get heard? Why does that matter?
5. **What changed with real data?** Compare to the toy data version of this exercise (if you've done it). Real reviews are messier — was Gemini better or worse on real data?

## Part 12 — Wrap-up

**What we did today:**

1. Downloaded **real Amazon reviews** from a public research dataset.
2. Did thematic analysis **by hand**, then compared with Gemini's output.
3. Built three measures: **coverage gap**, **rank correlation**, **subgroup disparity**.
4. Sent feedback back to Gemini and watched what improved and what got worse.

**The key idea from the paper:**

> Reading the generated themes alone, you can't tell if the AI is doing well. The metrics are what surface the problem.

The framework's main value is **diagnostic** — it doesn't fix the problem on its own, but it tells you exactly what is going wrong.

## Homework / Extensions

- Change `SEED` at the top to a different number and re-run. Do the themes shift?
- Increase `REVIEWS_PER_CATEGORY` to 20 and see whether Gemini gets better or worse.
- Pull from a different category (`Beauty_and_Personal_Care`, `Toys_and_Games`, etc.) and see if Gemini's behavior changes.
- Run several rounds of feedback. Does coverage keep improving? Does ordering keep degrading?

---

**Paper citation**

Im, T., & Park, H. (2026). *Improving LLM Thematic Analysis through Metric-Driven Self-Correction.* FLAIRS-39.

**Data citation**

Hou, Y., Li, J., He, Z., Yan, A., Chen, X., & McAuley, J. (2024). *Bridging Language and Items for Retrieval and Recommendation.* arXiv:2403.03952. [HuggingFace dataset: `McAuley-Lab/Amazon-Reviews-2023`]